<a href="https://colab.research.google.com/github/Sebi2005/Metaheuristics/blob/main/notebooks/AntColonyOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving fi10639.tsp to fi10639.tsp
Saving kroC100.tsp to kroC100.tsp


**Core Logic: Ant System (AS) for TSP**

In AS, a group of ants constructs solutions by moving between cities. Their choice is guided by:  

Pheromone Intensity ($\tau_{ij}$): How many ants traveled this path before.

Heuristic Information ($\eta_{ij}$): The visibility (usually $1 / \text{distance}$), favoring closer cities.

**The Probability Formula**

The probability of ant $k$ moving from city $i$ to city $j$ is:$$P_{ij}^k = \frac{[\tau_{ij}]^\alpha \cdot [\eta_{ij}]^\beta}{\sum_{l \in \text{allowed}} [\tau_{il}]^\alpha \cdot [\eta_{il}]^\beta}$$

**Implmentation**

**a. Initialization**

We need a distance matrix, a pheromone matrix (initialized to a small constant), and 10 ants.

In [2]:
import random
import numpy as np

def initialize_aco(num_cities, initial_pheromone=0.1):
    pheromones = np.full((num_cities, num_cities), initial_pheromone)
    return pheromones

def get_dist_matrix(cities):
    num_cities = len(cities)
    dist_matrix = np.zeros((num_cities, num_cities))
    for i in range(num_cities):
        for j in range(num_cities):
            dist_matrix[i][j] = np.linalg.norm(np.array(cities[i]) - np.array(cities[j]))
    return dist_matrix

We use this function to calculate the distance when we use the instance with 10000 data so we optimize the memory. A matrix with 10000 x 10000 dimension would cause a memory overflow.

In [10]:
def get_distance(c1,c2):
  return ((c1[0] -c2[0])**2 + (c1[1]-c2[1])**2)**0.5

**b.**

One Iteration for 10 Ants

Each ant builds a full tour, then we update pheromones based on the quality of those tours.

In [ ]:
def run_one_iteration(num_ants, dist_matrix, pheromones, alpha=1, beta=2):
    num_cities = len(dist_matrix)
    all_tours = []
    all_lengths = []

    for ant in range(num_ants):
        start_city = random.randint(0, num_cities - 1)
        tour = [start_city]
        visited = {start_city}

        while len(tour) < num_cities:
            i = tour[-1]
            probs = []
            for j in range(num_cities):
                if j not in visited:
                    tau = pheromones[i][j] ** alpha
                    eta = (1.0 / dist_matrix[i][j]) ** beta
                    probs.append(tau * eta)
                else:
                    probs.append(0)

            prob_sum = sum(probs)
            probs = [p / prob_sum for p in probs]
            next_city = np.random.choice(range(num_cities), p=probs)

            tour.append(next_city)
            visited.add(next_city)

        all_tours.append(tour)
        length = sum(dist_matrix[tour[k]][tour[k+1]] for k in range(num_cities - 1))
        length += dist_matrix[tour[-1]][tour[0]]
        all_lengths.append(length)

    return all_tours, all_lengths

**c. Pheromone Update (Evaporation + Deposit)**

Pheromones evaporate over time, and ants deposit new pheromones based on how short their tour was.

$$\tau_{ij} = (1 - \rho) \cdot \tau_{ij} + \sum \Delta \tau_{ij}$$

In [ ]:
def update_pheromones(pheromones, tours, lengths, rho=0.5):
    pheromones *= (1 - rho)

    for tour, length in zip(tours, lengths):
        contribution = 1.0 / length
        for k in range(len(tour) - 1):
            pheromones[tour[k]][tour[k+1]] += contribution
            pheromones[tour[k+1]][tour[k]] += contribution

Execution: Determine Global Best after 10 Iterations.

In [ ]:
def lab8_work(cities, num_ants=10, iterations=10,alpha=1.0, beta=2.0, rho=0.5):
    dist_matrix = get_dist_matrix(cities)
    pheromones = initialize_aco(len(cities))

    global_best_length = float('inf')
    global_best_tour = None

    for i in range(iterations):
        tours, lengths = run_one_iteration(num_ants, dist_matrix, pheromones,alpha, beta)
        update_pheromones(pheromones, tours, lengths,rho)

        min_len = min(lengths)
        if min_len < global_best_length:
            global_best_length = min_len
            global_best_tour = tours[lengths.index(min_len)]

        print(f"Iteration {i+1}: Best length = {min_len:.2f}")

    return global_best_tour, global_best_length

In [ ]:
def read_tsp_file(filename):
    locations = []
    reading_coords = False
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("NODE_COORD_SECTION"):
                reading_coords = True
                continue
            if line == "EOF" or not line: break
            if reading_coords:
                parts = line.split()
                locations.append((float(parts[1]), float(parts[2])))
    return locations

my_cities = read_tsp_file("kroC100.tsp")

best_tour, best_len = lab8_work(my_cities, num_ants=10, iterations=10)

Iteration 1: Best length = 51717.25
Iteration 2: Best length = 54458.69
Iteration 3: Best length = 56482.87
Iteration 4: Best length = 55988.42
Iteration 5: Best length = 55278.68
Iteration 6: Best length = 56665.77
Iteration 7: Best length = 52196.17
Iteration 8: Best length = 54922.17
Iteration 9: Best length = 52777.74
Iteration 10: Best length = 50795.41


The performance of the Ant System (AS) is highly sensitive to three main parameters: $\alpha$ (pheromone importance), $\beta$ (heuristic/distance importance), and $\rho$ (evaporation rate).  

 Key Parameters to Test

 $\alpha$ (Alpha): Controls the influence of the pheromone trails. A higher value makes ants follow previously discovered "good" paths more strictly.  

 $\beta$ (Beta): Controls the influence of the distance. A higher value makes ants more "greedy," preferring the closest available city.  

 $\rho$ (Rho): The evaporation rate. This prevents the algorithm from converging too quickly on a sub-optimal path by "forgetting" old pheromones

In [ ]:
def test_aco_parameters(cities):
    test_configs = [
        {"name": "Balanced", "alpha": 1.0, "beta": 2.0, "rho": 0.5},
        {"name": "Greedy (High Beta)", "alpha": 1.0, "beta": 5.0, "rho": 0.5},
        {"name": "Strong Pheromone (High Alpha)", "alpha": 5.0, "beta": 2.0, "rho": 0.5},
        {"name": "Fast Evaporation (High Rho)", "alpha": 1.0, "beta": 2.0, "rho": 0.8}
    ]

    print(f"{'Config Name':<30} | {'Global Best Length':<20}")
    print("-" * 55)

    for cfg in test_configs:
        _, best_length = lab8_work(
            cities,
            num_ants=10,
            iterations=10,
            alpha=cfg['alpha'],
            beta=cfg['beta'],
            rho=cfg['rho']
        )
        print(f"{cfg['name']:<30} | {best_length:<20.2f}")

test_aco_parameters(my_cities)

Config Name                    | Global Best Length  
-------------------------------------------------------
Iteration 1: Best length = 53722.73
Iteration 2: Best length = 56481.79
Iteration 3: Best length = 50134.82
Iteration 4: Best length = 54682.59
Iteration 5: Best length = 50379.19
Iteration 6: Best length = 53688.86
Iteration 7: Best length = 52769.91
Iteration 8: Best length = 54295.57
Iteration 9: Best length = 50756.50
Iteration 10: Best length = 50543.26
Balanced                       | 50134.82            
Iteration 1: Best length = 29268.48
Iteration 2: Best length = 28279.88
Iteration 3: Best length = 29602.02
Iteration 4: Best length = 28062.27
Iteration 5: Best length = 27199.69
Iteration 6: Best length = 30543.54
Iteration 7: Best length = 30434.43
Iteration 8: Best length = 28640.62
Iteration 9: Best length = 28821.03
Iteration 10: Best length = 28722.19
Greedy (High Beta)             | 27199.69            
Iteration 1: Best length = 56006.54
Iteration 2: Best length

In [8]:
def read_tsp_file(filename):
    locations = []
    reading_coords = False
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("NODE_COORD_SECTION"):
                reading_coords = True
                continue
            if line == "EOF" or not line: break
            if reading_coords:
                parts = line.split()
                locations.append((float(parts[1]), float(parts[2])))
    return locations

kro_cities = read_tsp_file("kroC100.tsp")
fi_cities = read_tsp_file("fi10639.tsp")


**Transition from Ant System (AS) to Ant Colony System (ACS)**


The primary goal of the transition is to improve the efficiency of the search and force the colony to converge more effectively on global optima. While AS is the basic variant where every ant contributes to pheromone updates, ACS introduces three sophisticated mechanisms.

**1. State Transition Rule (Pseudorandom Proportional)**

In AS, ants always choose the next city based on a probability distribution. In ACS, we introduce a parameter $q_0$ that allows the ant to choose between Exploitation and Exploration.

Exploitation: With probability $q_0$, the ant greedily chooses the absolute best edge (highest $\tau \cdot \eta$).

Exploration: With probability $(1-q_0)$, the ant uses the standard probabilistic rule to discover new paths.

In [11]:
def acs_transition_rule(i, visited, pheromones,cities, alpha, beta, q0):
    num_cities = len(cities)
    unvisited = [j for j in range(num_cities) if j not in visited]
    attractiveness = []
    for j in unvisited:
        tau = pheromones[i][j] ** alpha
        eta = (1.0 / (get_distance(cities[i],cities[j]) + 1e-9)) ** beta
        attractiveness.append(tau * eta)

    if random.random() < q0:
        return unvisited[np.argmax(attractiveness)]
    else:
        prob_sum = sum(attractiveness)
        probs = [a / prob_sum for a in attractiveness]
        return np.random.choice(unvisited, p=probs)

**2. Local Pheromone Update (The "Eating" Rule)**

This is a unique feature of ACS that happens during the tour construction, not at the end.

The Mechanism: As an ant crosses an edge $(i, j)$, it immediately removes a small amount of pheromone from that edge.

The Goal: By making a recently traveled edge less attractive, it encourages subsequent ants in the same iteration to explore different paths. This maintains diversity and prevents the entire colony from following a single ant into a local optimum within a single generation.

In [5]:
def local_pheromone_update(pheromones, i, j, rho_local, tau0):
    pheromones[i][j] = (1 - rho_local) * pheromones[i][j] + rho_local * tau0
    pheromones[j][i] = pheromones[i][j]

**3. Global Pheromone Update (Best-Only Reinforcement)**


In AS, all ants deposit pheromones at the end of an iteration. This can "blur" the search if many ants find mediocre paths.

The ACS Change: Only the Global Best ant (the one that found the shortest tour since the start of the trial) is allowed to deposit pheromones.

The Impact: This provides a much stronger "trail" for the rest of the colony to follow, leading to faster convergence on high-quality solutions.

In [6]:
def global_pheromone_update(pheromones, best_tour, best_length, rho_global):
    contribution = 1.0 / best_length
    for k in range(len(best_tour) - 1):
        i, j = best_tour[k], best_tour[k+1]
        pheromones[i][j] = (1 - rho_global) * pheromones[i][j] + rho_global * contribution
        pheromones[j][i] = pheromones[i][j]

By putting these 3 mechanism together we can create our main function.

In [12]:
def run_acs_assignment(cities, pop_size=10, gens=50, alpha=1, beta=2, rho_g=0.1, rho_l=0.1, q0=0.9):
    num_cities = len(cities)
    tau0 = 1.0 / (num_cities * 50000)
    pheromones = np.full((num_cities, num_cities), tau0)

    global_best_tour = None
    global_best_length = float('inf')

    for gen in range(gens):
        tours = []
        lengths = []

        for ant in range(pop_size):
            curr = random.randint(0, num_cities - 1)
            tour = [curr]
            visited = {curr}

            while len(tour) < num_cities:
                next_city = acs_transition_rule(curr, visited, pheromones, cities, alpha, beta, q0)
                local_pheromone_update(pheromones, curr, next_city, rho_l, tau0)
                tour.append(next_city)
                visited.add(next_city)
                curr = next_city
            tour_len = 0
            for i in range(len(tour)-1):
              tour_len += get_distance(cities[tour[i]],cities[tour[i+1]])
            tour_len+= get_distance(cities[tour[-1]],cities[tour[0]])
            tours.append(tour)
            lengths.append(tour_len)

            if tour_len < global_best_length:
                global_best_length = tour_len
                global_best_tour = tour

        global_pheromone_update(pheromones, global_best_tour, global_best_length, rho_g)

    return global_best_length

We run it on different parameters to see which one has the best results on our instances.

In [13]:
import csv
def run_comprehensive_A8_experiments(kro_cities, fi_cities):
    all_results = []

    kro_test_configs = [
        {"name": "Standard ACS", "q0": 0.9, "beta": 2.0, "rho": 0.1},
        {"name": "Exploratory", "q0": 0.5, "beta": 2.0, "rho": 0.1},
        {"name": "Greedy", "q0": 0.9, "beta": 5.0, "rho": 0.1},
        {"name": "High Evap", "q0": 0.9, "beta": 2.0, "rho": 0.5}
    ]

    for cfg in kro_test_configs:
        print(f"Running kroC100: {cfg['name']}...")
        best_l = run_acs_assignment(
            kro_cities, pop_size=10, gens=20,
            q0=cfg['q0'], beta=cfg['beta'], rho_g=cfg['rho']
        )
        all_results.append({
            "Instance": "kroC100", "Config": cfg['name'],
            "q0": cfg['q0'], "Beta": cfg['beta'], "Rho": cfg['rho'],
            "Best_Length": best_l
        })

    fi_test_configs = [
        {"name": "Greedy ACS", "q0": 0.98, "beta": 5.0, "rho": 0.1},
        {"name": "Balanced", "q0": 0.70, "beta": 2.0, "rho": 0.1}
    ]

    for cfg in fi_test_configs:
        print(f"Running fi10639: {cfg['name']}...")
        best_l = run_acs_assignment(
            fi_cities, pop_size=2, gens=2,
            q0=cfg['q0'], beta=cfg['beta'], rho_g=cfg['rho']
        )
        all_results.append({
            "Instance": "fi10639", "Config": cfg['name'],
            "q0": cfg['q0'], "Beta": cfg['beta'], "Rho": cfg['rho'],
            "Best_Length": best_l
        })

    with open('A8_Detailed_Comparison.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=all_results[0].keys())
        writer.writeheader()
        writer.writerows(all_results)

run_comprehensive_A8_experiments(kro_cities, fi_cities)

Running kroC100: Standard ACS...
Running kroC100: Exploratory...
Running kroC100: Greedy...
Running kroC100: High Evap...
Running fi10639: Greedy ACS...
Running fi10639: Balanced...


|Instance|Config|q0|Beta|Rho|Best\_Length|
|---|---|---|---|---|---|
|kroC100|Standard ACS|0\.9|2\.0|0\.1|23866\.45513320099|
|kroC100|Exploratory|0\.5|2\.0|0\.1|29786\.164333905603|
|kroC100|Greedy|0\.9|5\.0|0\.1|22241\.95101446071|
|kroC100|High Evap|0\.9|2\.0|0\.5|21388\.838844050882|
|fi10639|Greedy ACS|0\.98|5\.0|0\.1|650036\.0644892437|
|fi10639|Balanced|0\.7|2\.0|0\.1|1778898\.528602657|



**1. Analysis of kroC100 Results**

For a 100-city instance, the best result (**21,388.84**) is very strong.  

*High Evaporation (Rho 0.5)*: This configuration performed the best for this instance (21,388.84). This suggests that clearing out old pheromone trails quickly allowed the ants to avoid local optima and find a better global path in fewer iterations.  

*Greedy (Beta 5.0)*: This also performed very well (22,241.95). A high $\beta$ value makes the ants prioritize the very nearest neighbors, which is a powerful heuristic for smaller TSP instances.  

*Exploratory (q0 0.5)*: This performed the worst for this instance (29,786.16). Since running only 20 generations, the "Exploratory" ants likely spent too much time trying random paths and didn't have enough time to converge on a high-quality solution.


**2. Analysis of fi10639 Results**

For the massive 10,639-city instance, the results show a massive gap between configurations.  

Greedy ACS (650,036.06) vs. Balanced (1,778,898.53): The **Greedy** configuration is nearly **3 times better**.  

The Scale Factor: In a 10,000+ city problem with only 2 generations, pheromones haven't had time to build up. Therefore, the algorithm relies almost entirely on the distance heuristic.  

Why Greedy Won: The high $q_0$ (0.98) and $\beta$ (5.0) forced the ants to pick the nearest available city. In such a large map, picking the nearest neighbor is a much more effective strategy than picking a random city, which is what the "Balanced" (q0 0.7) setting occasionally does.